# Proyecto Analítico – Google Play Store

## Objetivo del Análisis

El propósito de este estudio es identificar los factores que influyen en la calificación (Rating) de una aplicación dentro de Google Play Store.

Se evaluarán variables estructurales como:

- Categoría
- Número de instalaciones
- Tamaño de la aplicación
- Precio
- Tipo (Gratis vs Paga)
- Sentimiento de usuarios

Hipótesis principal:
La calificación de una aplicación está influenciada por características estructurales del producto y por la percepción emocional de los usuarios.

Este análisis busca generar recomendaciones estratégicas basadas en datos para maximizar rating, posicionamiento y potencial de monetización.


## Metodología

El análisis se desarrolló bajo un enfoque exploratorio (EDA – Exploratory Data Analysis), siguiendo las siguientes etapas:

1. Limpieza y transformación de datos.
2. Análisis descriptivo univariado.
3. Análisis bivariado y correlacional.
4. Evaluación comparativa por segmentos (Categoría y Tipo).
5. Integración de análisis de sentimiento.

Se utilizaron métricas estadísticas básicas (media, correlación de Pearson) y visualización de datos para identificar patrones relevantes.


## 1. Google Play Store apps and reviews
<p>Las aplicaciones móviles están en todas partes. Son fáciles de crear y pueden resultar muy lucrativas. Debido a estos dos factores, se están desarrollando cada vez más aplicaciones. En este ejercicio, haremos un análisis completo del mercado de aplicaciones de Android comparando más de diez mil aplicaciones en Google Play en diferentes categorías. Buscaremos información valiosa en los datos para diseñar estrategias que impulsen el crecimiento y la retención.</p>
<p><img src="https://assets.datacamp.com/production/project_619/img/google_play_store.png" alt="Google Play logo"></p>
<p>Tenemos dos fuentes de datos:</p>
<ul>
<li><code>apps.csv</code>: contiene todos los detalles de las aplicaciones en Google Play. Hay 13 características que describen una aplicación determinada.</li>
<li><code>user_reviews.csv</code>: contiene 100 reseñas para cada aplicación, <a href="https://www.androidpolice.com/2019/01/21/google-play-stores-redesigned-ratings-and-reviews-section-lets-you-easily-filter-by-star-rating/">reviews</a>. El texto de cada reseña se ha procesado previamente y se le atribuyen tres características nuevas: Sentimiento (positivo, negativo o neutral), Polaridad del sentimiento y Subjetividad del sentimiento..</li>
</ul>

In [ ]:
# ================================
# 1. Carga inicial del dataset
# ================================

# Importar librerías
import pandas as pd
import matplotlib.pyplot as plt

# Cargar dataset
apps = pd.read_csv("../data26part1/apps.csv")

# Dimensión original
print("Dimensión original del dataset:", apps.shape)

# Eliminar duplicados
apps = apps.drop_duplicates()

# Dimensión después de eliminar duplicados
print("Dimensión después de eliminar duplicados:", apps.shape)

# Estadística descriptiva
print("\nResumen estadístico:")
print(apps.describe())

# Vista preliminar del dataset
print("\nPrimeras filas del dataset:")
print(apps.head())


In [ ]:
# ================================
# Asegurar que Rating sea numérico
# ================================

apps['Rating'] = pd.to_numeric(apps['Rating'], errors='coerce')

print("Tipo de dato de Rating:")
print(apps['Rating'].dtype)


In [ ]:
# ================================
# Inspección técnica del dataset
# ================================

print("Información general del dataset:")
apps.info()

print("\nValores nulos por columna:")
print(apps.isnull().sum())


## 2. Data cleaning
<p>Las cuatro variables con las que trabajaremos con más frecuencia de ahora en adelante son <i>Installs</i>, <i>Size</i>, <i>Rating</i> y <i>Price</i>. La función <code>info()</code> nos dice que las columnas <i>Installs</i> y <i>Price</i> son de tipo <code>object</code>, no son de tipo <code>int</code> o <code>float</code> como esperaríamos. Esto se debe a que la columna contiene algunos caracteres más que solo [0,9] dígitos. Idealmente, queremos que estas columnas fueran puramente numéricas<br>
<br>
Por lo tanto, ahora necesitamos limpiar nuestros datos. Específicamente, los caracteres especiales <code>,</code> y <code>+</code> que se encuentran en la columna <i>Installs</i> y <code>$</code> que esta en la columna <i>Price</i>.</p>


In [ ]:
# Lista de caracteres a eliminar
chars_to_remove = [',', '+', '$']

# Lista de las columnas a limpiar
cols_to_clean = ['Installs', 'Price']

for col in cols_to_clean:
    apps[col] = (
        apps[col]
        .astype(str)
        .str.replace(r'[,+$]', '', regex=True)
        .astype(float)
    )

apps['Installs'] = apps['Installs'].astype(int)



## Verificación de Tipos de Datos Después de la Limpieza

En esta sección se valida que las columnas **Installs** y **Price** 
fueron correctamente transformadas a tipos numéricos tras la eliminación 
de caracteres especiales como ',', '+' y '$'.

Esta verificación es fundamental porque:

- Asegura que la limpieza fue exitosa.
- Confirma que no quedaron valores tipo string ocultos.
- Garantiza que el dataset está listo para análisis estadístico.
- Previene errores en cálculos de correlación o modelado posterior.

Tipos esperados:
- **Installs → int** (conteo de descargas)
- **Price → float** (valor monetario)


In [ ]:
# ================================
# Verificación técnica de tipos de datos
# ================================

print("\nTipos de datos después de limpieza:")
print(apps[['Installs', 'Price']].dtypes)

print("\nResumen estadístico posterior a limpieza:")
print(apps[['Installs', 'Price']].describe())


## 3. Exploring App's categories
<p>Con más de mil millones de usuarios activos en 190 países de todo el mundo, Google Play sigue siendo una importante plataforma de distribución para crear una audiencia global. Para que las empresas muestren sus aplicaciones a los usuarios, es importante hacerlas más rápida y fácilmente visibles en Google Play. Para mejorar la experiencia de búsqueda general, Google ha introducido el concepto de agrupar aplicaciones en categorías.</p>
<p>Esto nos lleva a las siguientes preguntas:</p>
<ul>
<li>¿Qué categoría tiene la mayor participación de aplicaciones (activas) en el mercado?</li>
<li>¿Alguna categoría específica domina el mercado?</li>
<li>¿Qué categorías tienen la menor cantidad de aplicaciones?</li>
</ul>
<p>Vamos a responder estas preguntas aquí <code>33</code> categorías unicas estan presentas en nuestro dataset. Las apps de <em>Family</em> y <em>Game</em> tienen la mayor prevalencia del mercado. Curiosamente, <em>Tools</em>, <em>Business</em> y <em>Medical</em> también están en el top.</p>

In [ ]:
# Imprime el total de categorías únicas
num_categories = apps['Category'].nunique()
print('Number of categories = ', num_categories)

# Cuenta el número de aplicaciones en cada Categoría y ordena de manera descendente 
num_apps_in_category = apps['Category'].value_counts()

# Muestra el resultado en una gráfica de barras
num_apps_in_category.plot(kind='bar', figsize=(12,6))
plt.title("Número de aplicaciones por categoría")
plt.xlabel("Categoría")
plt.ylabel("Cantidad de aplicaciones")
plt.tight_layout()
plt.show()



### Conclusión Estratégica – Distribución por Categorías

El análisis muestra que existen 33 categorías activas en el dataset, lo que confirma un mercado altamente segmentado y competitivo.

Las categorías *Family* y *Game* concentran la mayor cantidad de aplicaciones, lo que sugiere:
- Alta demanda del mercado.
- Bajo costo de entrada para desarrolladores.
- Fuerte competencia interna.

Sin embargo, una alta cantidad de aplicaciones no implica necesariamente mayor rentabilidad. Categorías como *Business*, *Tools* y *Medical*, aunque con menor volumen, podrían representar mercados más especializados y con menor saturación.

Desde una perspectiva estratégica:
- Entrar en categorías saturadas requiere diferenciación clara y fuerte inversión en marketing.
- Categorías menos saturadas pueden ofrecer nichos más rentables si la propuesta de valor es sólida.

Este análisis sugiere que la decisión de desarrollar una aplicación debe considerar no solo popularidad, sino también competencia y potencial de monetización.


## 4. Ratings Distribution
<p>Después de analizar la participación de mercado para cada categoría de las aplicaciones, veamos cómo se posicionan de acuerdo a las calificaciones (en una escala del 1 al 5) las cuales afectan la imagen de la marca general de la empresa. Las calificaciones son un indicador clave de rendimiento de una aplicación.</p>

### Manejo de Valores Nulos en Rating

Antes de realizar cualquier análisis estadístico sobre la variable Rating,
se cuantifican los valores faltantes para evaluar su impacto en el dataset.

Dado que Rating es una variable central del estudio,
se eliminarán únicamente las filas donde esta variable sea nula.


### Verificación inicial de integridad de la variable Rating

Se calcula el número total de valores nulos en la variable Rating
y la dimensión actual del dataset antes de realizar la eliminación.

Esta verificación permite cuantificar el impacto de la limpieza
sobre el tamaño total del conjunto de datos.


In [ ]:
# Diagnóstico preliminar de integridad de datos (Rating)
print("Cantidad de ratings nulos antes de eliminar:",
      apps['Rating'].isnull().sum())

print("Dimensión antes de eliminar nulos:", apps.shape)


In [ ]:
# ==========================================
# Analizar impacto de valores nulos Rating
# ==========================================

null_count = apps['Rating'].isnull().sum()
total_count = len(apps)

print("Cantidad de ratings nulos:", null_count)
print("Porcentaje de ratings nulos:",
      round(null_count / total_count * 100, 2), "%")


In [ ]:
# Eliminación de ratings nulos
apps = apps.dropna(subset=['Rating'])

print("Dimensión después de eliminar nulos:", apps.shape)

# Calcular el promedio de calificación de las apps
avg_app_rating = apps['Rating'].mean()
print('Average app rating = ', avg_app_rating)

# Calcula el promedio de calificación por categoría
print(apps.groupby('Category')['Rating'].mean().sort_values(ascending=False))

# Visualiza en un histograma el comportamiento del Rating
apps['Rating'].hist(bins=20)
plt.title("Distribución de Ratings")
plt.xlabel("Rating")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()



### Conclusión Estratégica – Distribución de Ratings

La mayoría de las aplicaciones presentan calificaciones superiores a 4.0, lo que indica un sesgo positivo en la evaluación de los usuarios.

Sin embargo, este fenómeno puede estar influenciado por:
- Autoselección de usuarios satisfechos.
- Eliminación temprana de apps mal calificadas del mercado.
- Estrategias de incentivo para dejar reseñas positivas.

Desde una perspectiva estratégica, mantener un rating superior a 4.0 parece ser un estándar competitivo mínimo dentro del ecosistema Google Play.

Esto implica que la experiencia de usuario, estabilidad y soporte post-lanzamiento son factores críticos para la sostenibilidad del producto.


## 5. Size and Price
<p>Examinemos ahora el tamaño y el precio de la aplicación. En cuanto al tamaño, si la aplicación móvil es demasiado grande, puede ser difícil y/o costoso para los usuarios descargarla. Los tiempos de descarga prolongados pueden desanimar a los usuarios incluso antes de que experimenten su aplicación móvil. Además, el dispositivo de cada usuario tiene una cantidad limitada de espacio en disco. Por el precio, algunos usuarios esperan que sus aplicaciones sean gratuitas o económicas. Estos problemas se agravan si el mercado objetivo es en países en vías de desarrollo; especialmente debido a las velocidades de Internet, el poder adquisitivo, los tipos de cambio, etc.</p>
<p>How can we effectively come up with strategies to size and price our app?</p>
<ul>
<li>¿El tamaño de una aplicación afecta su calificación?</li>
<li>¿Los usuarios realmente se preocupan por las aplicaciones pesadas del sistema o prefieren las aplicaciones ligeras? </li>
<li>¿El precio de una aplicación afecta su calificación? </li>
<li>¿Los usuarios siempre prefieren las aplicaciones gratuitas a las de paga?</li>
</ul>

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")
sns.set_style("darkgrid")

# 🔹 Asegurar que Size sea string antes de manipular texto
apps['Size'] = apps['Size'].astype(str)

# Reemplazar 'Varies with device' por NaN
apps['Size'] = apps['Size'].replace('Varies with device', pd.NA)

# Eliminar letra 'M'
apps['Size'] = apps['Size'].str.replace('M', '', regex=False)

# Convertir a numérico
apps['Size'] = pd.to_numeric(apps['Size'], errors='coerce')

# Filtrar filas donde Rating y Size no sean nulos
apps_clean = apps.dropna(subset=['Rating', 'Size'])

# Filtrar categorías con al menos 250 apps
large_categories = (
    apps_clean
    .groupby('Category')
    .filter(lambda x: len(x) >= 250)
)

# 🔹 1️⃣ Relación Tamaño vs Rating
g1 = sns.jointplot(
    data=large_categories,
    x='Size',
    y='Rating',
    kind='scatter'
)

g1.fig.suptitle("Relación entre Tamaño de la App (MB) y Rating", y=1.02)

# 🔹 2️⃣ Relación Precio vs Rating (Apps de Paga)
paid_apps = apps_clean[apps_clean['Type'] == 'Paid']

g2 = sns.jointplot(
    data=paid_apps,
    x='Price',
    y='Rating',
    kind='scatter'
)

g2.fig.suptitle("Relación entre Precio y Rating (Apps de Paga)", y=1.02)

plt.show()


### Transformación Logarítmica de Installs

La variable **Installs** presenta una distribución altamente asimétrica positiva,
debido a la presencia de aplicaciones con millones de descargas frente a otras con pocas instalaciones.

Para reducir el impacto de valores extremos y estabilizar la varianza,
se aplica una transformación logarítmica utilizando `log1p()`.

Esta transformación permite:

- Mejorar la interpretación de correlaciones.
- Reducir distorsión por outliers.
- Aproximar la variable a una distribución más normal.


In [ ]:
# ================================
# Matriz de correlación
# ================================

import numpy as np

apps['Log_Installs'] = np.log1p(apps['Installs'])

numeric_cols = ['Rating', 'Size', 'Log_Installs', 'Price']

correlation_matrix = apps[numeric_cols].dropna().corr(method='pearson')

print("Matriz de correlación:")
print(correlation_matrix)

plt.figure(figsize=(8,6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


### Conclusión Analítica – Correlación

La matriz de correlación confirma que no existe una relación lineal fuerte entre Rating y las variables Size o Price.

Esto sugiere que la calificación depende más de factores cualitativos como experiencia de usuario, estabilidad y propuesta de valor, que de variables puramente estructurales.

Las instalaciones muestran correlación moderada con Rating, lo cual puede indicar que aplicaciones con mayor adopción tienden a consolidar reputación.


### Conclusión Estratégica – Tamaño y Precio

El análisis sugiere que no existe una relación lineal fuerte entre tamaño y calificación.

Esto indica que los usuarios priorizan funcionalidad y experiencia sobre peso del archivo.

En cuanto al precio, las aplicaciones de paga no presentan una penalización significativa en rating promedio, lo que sugiere que los usuarios están dispuestos a pagar si perciben valor real.

Desde una perspectiva estratégica, el precio debe alinearse con propuesta de valor y segmento objetivo.


## 6. Relation between Category & Price
<p>Así que ahora viene la parte difícil. ¿Cómo se supone que las empresas y los desarrolladores cubran sus cuotas de fin de mes? ¿Qué estrategias de monetización pueden utilizar las empresas para maximizar las ganancias? Los costos de las aplicaciones se basan en gran medida en las características, la complejidad y la plataforma. Hay muchos factores a considerar al seleccionar la estrategia de precios adecuada para las aplicaciones moviles. Es importante considerar la disposición de su cliente a pagar por la aplicación. Un precio elevado puede hacer que los clientes no se vean atraídos por descargarlaque ocurra la descarga o pueden eliminar una aplicación que han descargado después de recibir demasiados anuncios o simplemente no obtener el valor que esperaban de su dinero.</p>

<p>Las diferentes categorías exigen diferentes rangos de precios. Algunas aplicaciones que son simples y se usan a diario, como la aplicación de calculadora, probablemente deberían mantenerse gratuitas. Sin embargo, tendría sentido cobrar por una aplicación médica altamente especializada que diagnostica a pacientes diabéticos, así que vamos a descubrir y encontrar la respuesta</p>

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
fig.set_size_inches(15, 8)

# Lista de categorías populares
popular_app_cats = apps[apps.Category.isin(['GAME', 'FAMILY', 'PHOTOGRAPHY',
                                            'MEDICAL', 'TOOLS', 'FINANCE',
                                            'LIFESTYLE','BUSINESS'])]

# Examina la tendencia de precio graficando el Precio por Categoría
ax = sns.stripplot(x = popular_app_cats['Category'], 
                   y = popular_app_cats['Price'], 
                   jitter=True, linewidth=1)

ax.set_title('App pricing trend across categories')

# ==========================================
# Detectar apps con precio mayor a 200 USD
# ==========================================

apps_above_200 = apps[apps['Price'] > 200]

print("Cantidad de apps con precio mayor a 200 USD:",
      len(apps_above_200))

apps_above_200.head()



### Conclusión Estratégica – Estrategia de Monetización

Se observa que ciertas categorías concentran aplicaciones de mayor precio, especialmente aquellas asociadas a productividad o servicios especializados.

Esto sugiere que la disposición a pagar está vinculada al tipo de necesidad que la aplicación satisface.

Categorías orientadas a entretenimiento tienden a modelos gratuitos o freemium, mientras que categorías profesionales pueden sostener precios más altos.


## 7. Paid apps vs Free apps
<p>Para las aplicaciones de Play Store en la actualidad, existen cinco tipos de estrategias de precios: gratis, "freemium", de pago, "paymium" y de suscripción. Centrémonos solo en aplicaciones gratuitas y de pago. 

Algunas características de las aplicaciones gratuitas son:</p>
<ul>
<li>Libres de descarga.</li>
<li>La principal fuente de ingresos a menudo proviene de la publicidad.</li>
<li>Por lo general son creadaa por empresas que tienen otros productos y la aplicación sirve como una extensión de esos productos.</li>
<li>Puede servir como una herramienta para la retención de clientes, la comunicación y el servicio al cliente.</li>
</ul>
<p>Algunas características de las aplicaciones de paga son:</p>
<ul>
<li>Tienen un tiempo de servicio de prueba gratuito, esto para que el usuario pueda conocerla.</li>
<li>Ofrecen un servicio de mayor especialidad.</li>
</ul>
<p>¿Además de esto que otras características diferencías a las aplicaciones de pago las aplicaciones gratuitas?</p>

In [ ]:
# ================================
# Análisis comparativo de métricas por Tipo (Free vs Paid)
# ================================

print("Average rating by Type:")
print(apps.groupby('Type')['Rating'].mean())

print("\nAverage installs by Type:")
print(apps.groupby('Type')['Installs'].mean())

print("\nAverage price by Type:")
print(apps.groupby('Type')['Price'].mean())


### Conclusión Estratégica – Apps Gratuitas vs de Paga

Las aplicaciones gratuitas presentan un volumen de descargas considerablemente mayor.

Sin embargo, las aplicaciones de paga muestran un rating promedio ligeramente superior.

Esto sugiere que los usuarios que pagan tienden a evaluar más favorablemente, posiblemente debido a una mayor expectativa de calidad y menor presencia de publicidad.

Desde el punto de vista de negocio, el modelo gratuito maximiza alcance, mientras que el modelo de paga prioriza ingreso por usuario.


## 8. Sentiment analysis
<p>La minería de datos de reseñas de usuarios para determinar cómo se sienten las personas acerca de su producto, marca o servicio se puede realizar mediante una técnica llamada análisis de sentimientos. Las reseñas de los usuarios de las aplicaciones se pueden analizar para identificar si el estado de ánimo es positivo, negativo o neutral con respecto a esa aplicación. Por ejemplo, las palabras positivas en la revisión de una aplicación pueden incluir palabras como "asombroso", "amigable", "bueno", "excelente" y "amor". Las palabras negativas pueden ser palabras como 'malware', 'odio', 'problema', 'reembolso' e 'incompetente'.</p>

<p>¿Qué podemos decir acerca del analisis de sentimiento de las aplicaciones?</p>

In [ ]:
# Carga el archivo user_reviews.csv
reviews_df = pd.read_csv("../data26part1/user_reviews.csv")

# Une los dos DataFrames (join)
merged_df = apps.merge(reviews_df, on='App', how='inner')

# Elimina los valores nulos (NA) de las columnas Sentiment y Review
merged_df = merged_df.dropna(subset = ['Sentiment', 'Review'])

# Grafica la polaridad de sentimientos para apps gratuitas y de paga
sns.set_style('ticks')
fig, ax = plt.subplots()
fig.set_size_inches(11, 8)

ax = sns.boxplot(x = merged_df['Type'], 
                 y = merged_df['Sentiment_Polarity'], 
                 data = merged_df)

ax.set_title('Sentiment Polarity Distribution')
plt.show()


In [ ]:
# ================================
# Distribución porcentual de Sentimientos
# ================================

print("Distribución porcentual de Sentimientos:")
print(merged_df['Sentiment'].value_counts(normalize=True))


### Conclusión Estratégica – Análisis de Sentimiento

El análisis de polaridad muestra que tanto aplicaciones gratuitas como de paga tienden a concentrar sentimientos positivos.

Sin embargo, la variabilidad en aplicaciones gratuitas es mayor, lo que sugiere:
- Mayor volumen de usuarios.
- Mayor diversidad en experiencias.
- Mayor exposición a críticas.

Las aplicaciones de paga muestran una distribución más estable, lo que podría indicar:
- Público más segmentado.
- Expectativas más claras.
- Mayor alineación entre valor percibido y precio.

Desde una perspectiva de negocio, el análisis de sentimiento complementa el rating tradicional, permitiendo evaluar calidad percibida más allá del promedio numérico.


## 9. Síntesis Ejecutiva

El análisis exploratorio evidencia que:

- La competencia es alta en categorías masivas como Game y Family.
- El mercado establece un estándar competitivo mínimo de rating > 4.0.
- El precio no penaliza necesariamente el rating si existe valor percibido.
- El sentimiento del usuario complementa el rating promedio como indicador de percepción real.

En términos estratégicos, el éxito de una aplicación depende menos de variables técnicas como tamaño y más de experiencia, segmentación y propuesta de valor clara.


## Limitaciones del Estudio

- El dataset representa una fotografía estática del mercado.
- No se incluyen variables temporales.
- La correlación no implica causalidad.
- El análisis de sentimiento depende de procesamiento previo del texto.

Futuros análisis podrían incorporar modelos predictivos o segmentación avanzada.


## 10. Conclusión Final del Análisis

A lo largo de este análisis exploratorio del mercado de aplicaciones en Google Play Store se identificaron patrones relevantes en términos de categorías, calificaciones, tamaño, precio y percepción del usuario.

### Resultados Cuantitativos Clave

El análisis se realizó sobre un total de 8,196 aplicaciones activas después del proceso de limpieza de datos.

El rating promedio global del mercado es 4.173, lo que confirma que el ecosistema presenta una tendencia positiva en la evaluación de los usuarios.

Al segmentar por modelo de negocio:

- Aplicaciones gratuitas: rating promedio de 4.166  
- Aplicaciones de paga: rating promedio de 4.260  

Esto indica que las aplicaciones de paga mantienen ligeramente mejores evaluaciones, posiblemente asociadas a mayor calidad percibida o segmentación de nicho.

El 92.63% del mercado está compuesto por aplicaciones gratuitas, lo que confirma que el modelo freemium domina claramente el ecosistema.

En términos de crecimiento y adopción, la correlación entre Rating y Log_Installs es de 0.085, lo que indica una relación positiva pero débil. Esto sugiere que una alta calificación no garantiza necesariamente un mayor volumen de descargas.

El análisis de sentimiento muestra la siguiente distribución:

- 64.22% reseñas positivas  
- 22.28% reseñas negativas  
- 13.50% reseñas neutrales  

La percepción general del mercado es predominantemente positiva, aunque existe un porcentaje relevante de experiencias negativas que podrían representar oportunidades de mejora.

### Implicaciones Estratégicas

Los hallazgos sugieren que:

- El mercado está altamente segmentado, con fuerte concentración en categorías masivas como Family y Game.
- Mantener un rating superior a 4.0 es prácticamente un requisito competitivo.
- El modelo gratuito domina el mercado, pero las apps de paga pueden competir con éxito si ofrecen alto valor percibido.
- La experiencia del usuario y la gestión de reputación digital son factores críticos para la sostenibilidad.

Este análisis no solo permite comprender el comportamiento actual del mercado, sino también diseñar estrategias fundamentadas en datos para el desarrollo de productos digitales competitivos.


## 11. Recomendaciones Ejecutivas

Desde una perspectiva de toma de decisiones estratégicas, se sugieren las siguientes líneas de acción:

### Estrategia de Monetización

Dado que el 92.63% del mercado es gratuito, ingresar con un modelo Paid requiere diferenciación clara.
Modelo freemium con monetización progresiva podría maximizar penetración inicial.

### Optimización de Experiencia de Usuario

El rating promedio competitivo (4.173) establece un umbral mínimo.
Se recomienda:

Testing continuo

Optimización de rendimiento

Respuesta activa a reseñas negativas

### Posicionamiento por Nicho

Las apps de paga muestran mejor rating promedio.
Esto sugiere oportunidades en segmentos especializados (productividad, finanzas, profesional).

### Gestión de Reputación

Con 22.28% de sentimiento negativo, existe riesgo reputacional.
Implementar análisis continuo de reviews puede anticipar crisis de percepción.

### Decisiones Basadas en Datos

La baja correlación entre instalaciones y rating indica que crecimiento en volumen no garantiza calidad percibida.
Las métricas de satisfacción deben medirse independientemente del crecimiento.